In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel,Field
import operator

In [ ]:
load_dotenv()

In [ ]:
model=ChatOpenAI(model="gpt-4o-mini")

In [ ]:
class EvaluationSchema(BaseModel):
    feedback:str=Field(description='Detailed feedback on the essay')
    score:int=Field(description='Score out of 1-',ge=0,le=10)
    

In [ ]:
evaluation_model=model.with_structured_output(EvaluationSchema)

In [ ]:
essay="""yo"""

In [ ]:
prompt=f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10\n {essay}'

In [ ]:
class UPSCState(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_scores:Annotated[list[int],operator.add]
    avg_score:float

In [ ]:
def language(state:UPSCState)->UPSCState:
    prompt=prompt=f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10\n {state['essay']}'
    output=evaluation_model.invoke(prompt)
    return{'language_feedback':output.feedback,'individual_scores':[output.score]}

In [ ]:
def analysis(state:UPSCState)->UPSCState:
    prompt=prompt=f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10\n {state['essay']}'
    output=evaluation_model.invoke(prompt)
    return{'analysis_feedback':output.feedback,'individual_scores':[output.score]}

In [ ]:
def thought(state:UPSCState)->UPSCState:
    prompt=prompt=f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10\n {state['essay']}'
    output=evaluation_model.invoke(prompt)
    return{'clarity_feedback':output.feedback,'individual_scores':[output.score]}

In [ ]:
def final_evaluation(state:UPSCState)->UPSCState:
    prompt=f'Based on the following feedbacks create a summarized feedback- language-{state['language_feedback']}\n,analysis-{state['analysis_feedback']}\n,thought-{state['clarity_feedback']}'
    output=model.invoke(prompt).content
    avg_score=sum(state['individual_scores']) / len(state['individual_scores'])
    return {'overall_feedback':output,'avg_score':avg_score}


In [ ]:
graph=StateGraph(UPSCState)
graph.add_node("evaluate_language",language)
graph.add_node('evaluate_analysis',analysis)
graph.add_node('evaluate_thought',thought)
graph.add_node('final_evaluation',final_evaluation)
graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_thought')
graph.add_edge('evaluate_language','final_evaluation')
graph.add_edge('evaluate_analysis','final_evaluation')
graph.add_edge('evaluate_thought','final_evaluation')
graph.add_edge('final_evaluation',END)
workflow=graph.compile()
graph.compile()

In [ ]:
initial_state={
    'essay':essay,
}
final_state=workflow.invoke(initial_state)